# 01 – Ingestion & Data-Quality Audit (Initial EDA)

**Project:** RavenStack SaaS Revenue & Subscription Analytics  
 initial EDA notebook  
**Data:** `data/raw/` (original files, never edited). Reaven Analytics (synthetic).

**How to use:** run the cells from top to bottom (*Run All*). Charts are saved to `reports/figures/`.


In [ ]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

# Find the repository root whether the notebook is run from /notebooks or from the root
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
RAW = ROOT / "data" / "raw"
FIG = ROOT / "reports" / "figures"
FIG.mkdir(parents=True, exist_ok=True)
DATA_END = pd.Timestamp("2024-12-31")
print("Repository root:", ROOT)

## 1. Load the raw tables

In [ ]:
accounts = pd.read_csv(RAW / "ravenstack_accounts.csv", parse_dates=["signup_date"])
subs = pd.read_csv(RAW / "ravenstack_subscriptions.csv", parse_dates=["start_date", "end_date"])
usage = pd.read_csv(RAW / "ravenstack_feature_usage.csv", parse_dates=["usage_date"])
tickets = pd.read_csv(RAW / "ravenstack_support_tickets.csv", parse_dates=["submitted_at", "closed_at"])
churn = pd.read_csv(RAW / "ravenstack_churn_events.csv", parse_dates=["churn_date"])

tables = {"accounts": accounts, "subscriptions": subs, "feature_usage": usage,
          "support_tickets": tickets, "churn_events": churn}
overview = pd.DataFrame({
    "rows": {k: len(v) for k, v in tables.items()},
    "columns": {k: v.shape[1] for k, v in tables.items()},
    "null cells": {k: int(v.isna().sum().sum()) for k, v in tables.items()},
})
overview

**Expected:** accounts 500, subscriptions 5,000, feature_usage 25,000, support_tickets 2,000, churn_events 600.

## 2. Missing values

In [ ]:
rows = []
for name, df in tables.items():
    for col, n in df.isna().sum().items():
        if n:
            rows.append({"table": name, "column": col, "nulls": n, "share": f"{n/len(df):.1%}"})
pd.DataFrame(rows)

**Reading this:** a null `end_date` means the subscription is still active, so it is not an error.
Null satisfaction scores mean no customer response; null feedback means no comment.

## 3. Keys and relationships

In [ ]:
checks = {
    "duplicate account_id": accounts.account_id.duplicated().sum(),
    "duplicate subscription_id": subs.subscription_id.duplicated().sum(),
    "duplicate usage_id (ID collisions)": usage.usage_id.duplicated().sum(),
    "duplicate ticket_id": tickets.ticket_id.duplicated().sum(),
    "duplicate churn_event_id": churn.churn_event_id.duplicated().sum(),
    "subscriptions with unknown account": (~subs.account_id.isin(accounts.account_id)).sum(),
    "usage with unknown subscription": (~usage.subscription_id.isin(subs.subscription_id)).sum(),
    "tickets with unknown account": (~tickets.account_id.isin(accounts.account_id)).sum(),
    "churn events with unknown account": (~churn.account_id.isin(accounts.account_id)).sum(),
}
pd.Series(checks, name="count").to_frame()

All keys link correctly (no orphans). The only identifier problem is **21 repeated `usage_id` values**
with different content, which we flag and keep rather than delete.

## 4. Subscription checks

In [ ]:
subs_sorted = subs.sort_values(["account_id", "start_date"]).copy()
subs_sorted["end_filled"] = subs_sorted.end_date.fillna(DATA_END)
subs_sorted["prev_end"] = subs_sorted.groupby("account_id").end_filled.shift()
overlap = (subs_sorted.start_date < subs_sorted.prev_end).sum()

same_month = subs[subs.end_date.notna() &
                  (subs.start_date.dt.to_period("M") == subs.end_date.dt.to_period("M"))]

sub_checks = {
    "subscriptions": len(subs),
    "subscriptions per account (average)": round(len(subs) / subs.account_id.nunique(), 1),
    "still active (no end_date)": int(subs.end_date.isna().sum()),
    "ended (have an end_date)": int(subs.end_date.notna().sum()),
    "trial subscriptions": int(subs.is_trial.sum()),
    "trial rows with MRR > 0": int(((subs.mrr_amount > 0) & subs.is_trial).sum()),
    "non-trial rows with MRR = 0": int(((subs.mrr_amount == 0) & ~subs.is_trial).sum()),
    "end_date before start_date": int((subs.end_date < subs.start_date).sum()),
    "arr_amount != 12 x mrr_amount": int((subs.arr_amount != 12 * subs.mrr_amount).sum()),
    "rows overlapping another subscription of the same account": int(overlap),
    "start and end in the same month": len(same_month),
    "upgrade AND downgrade flag both True": int((subs.upgrade_flag & subs.downgrade_flag).sum()),
}
pd.Series(sub_checks, name="value").to_frame()

**Key points for the team**

- Accounts hold many subscriptions at the same time, so MRR is a **sum across active subscriptions**, not one number per account.
- `arr_amount` is always 12 × `mrr_amount`, so `mrr_amount` is **already monthly** (do not divide annual plans again).
- Trials always carry MRR 0, so they are excluded from MRR but kept for trial analysis.
- Subscriptions that start and end in the same month never appear in a month-end snapshot. This is a known limitation.

## 5. Three conflicting churn sources

In [ ]:
flagged = set(accounts.loc[accounts.churn_flag, "account_id"])
with_event = set(churn.account_id)
active_sub = set(subs.loc[subs.end_date.isna(), "account_id"])

churn_sources = pd.DataFrame({
    "accounts": {
        "churn_flag = True (accounts table)": len(flagged),
        "have a churn event (churn_events table)": len(with_event),
        "flagged but NO churn event": len(flagged - with_event),
        "churn event but NOT flagged": len(with_event - flagged),
        "flagged as churned yet hold an active subscription": len(flagged & active_sub),
        "accounts with more than one churn event": int((churn.groupby("account_id").size() > 1).sum()),
    }
})
print("Subscriptions with an end_date:", int(subs.end_date.notna().sum()))
print("Reactivation events:", int(churn.is_reactivation.sum()))
churn_sources

**Team notes (write your conclusion here):** the three sources do not agree, so customer churn needs one
documented definition (see `docs/metric_definitions.md`) and the others reported as sensitivity checks.

## 6. Usage and tickets

In [ ]:
u = usage.merge(subs[["subscription_id", "start_date"]], on="subscription_id")
print("Usage rows dated before their subscription start:", int((u.usage_date < u.start_date).sum()),
      "of", len(u), f"({(u.usage_date < u.start_date).mean():.0%})")
print("Tickets closed before submitted:", int((tickets.closed_at < tickets.submitted_at).sum()))
print("Missing satisfaction scores:", int(tickets.satisfaction_score.isna().sum()))

## 7. Charts

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 9))

# 1. Signups per month
s = accounts.set_index("signup_date").resample("ME").size()
axes[0, 0].bar(s.index.strftime("%Y-%m"), s.values, color="#4C72B0")
axes[0, 0].set_title("New accounts per month")
axes[0, 0].tick_params(axis="x", rotation=90, labelsize=7)

# 2. Subscriptions by plan
p = subs.plan_tier.value_counts()
axes[0, 1].bar(p.index, p.values, color="#55A868")
axes[0, 1].set_title("Subscriptions by plan tier")

# 3. Billing frequency
b = subs.billing_frequency.value_counts()
axes[0, 2].bar(b.index, b.values, color="#C44E52")
axes[0, 2].set_title("Billing frequency")

# 4. Referral source
r = accounts.referral_source.value_counts()
axes[1, 0].bar(r.index, r.values, color="#8172B2")
axes[1, 0].set_title("Accounts by referral source")

# 5. Churn reasons
cr = churn.reason_code.value_counts()
axes[1, 1].barh(cr.index[::-1], cr.values[::-1], color="#CCB974")
axes[1, 1].set_title("Churn events by reason")

# 6. Tickets by priority
tp = tickets.priority.value_counts().reindex(["low", "medium", "high", "urgent"])
axes[1, 2].bar(tp.index, tp.values, color="#64B5CD")
axes[1, 2].set_title("Support tickets by priority")

plt.tight_layout()
plt.savefig(FIG / "01_eda_overview.png", dpi=150)
plt.show()

## 8. Naive month-end MRR (first look only)

A subscription counts at a month-end if it started on or before that date and has not ended by that date.
Trials are excluded. This is only a **first look**; the validated KPI tables come from the snapshot and the KPI analyst.

In [ ]:
month_ends = pd.date_range("2023-01-31", "2024-12-31", freq="ME")
mrr, n_active = [], []
for m in month_ends:
    active = subs[(subs.start_date <= m) & (subs.end_date.isna() | (subs.end_date > m)) & (~subs.is_trial)]
    mrr.append(active.mrr_amount.sum())
    n_active.append(len(active))
mrr_trend = pd.DataFrame({"month_end": month_ends, "active_paid_subscriptions": n_active, "mrr": mrr})

fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
ax[0].plot(mrr_trend.month_end, mrr_trend.mrr / 1e6, marker="o")
ax[0].set_title("Month-end MRR (USD millions, trials excluded)")
ax[1].plot(mrr_trend.month_end, mrr_trend.active_paid_subscriptions, marker="o", color="#DD8452")
ax[1].set_title("Active paid subscriptions at month-end")
plt.tight_layout()
plt.savefig(FIG / "01_mrr_first_look.png", dpi=150)
plt.show()

mrr_trend.loc[mrr_trend.month_end == "2024-06-30"]

**Benchmark:** June 2024 MRR should be **3,833,405** (this equals the validated snapshot).

**Why MRR only rises:** about 90% of subscriptions never end in this synthetic dataset, so new revenue
always outweighs churn. This is a property of the data, not a calculation error.

## 9. Summary of findings (edit as a team)

1. All 5 tables load with the expected row counts and no orphan keys.
2. Accounts hold ~10 overlapping subscriptions each; MRR must be summed across active subscriptions.
3. `mrr_amount` is already monthly; trials carry 0 MRR.
4. About 90% of subscriptions never end, so MRR rises almost every month.
5. No acquisition spend exists, so CAC cannot be calculated.

